# Test Phenom-Beta

Test out PhenomBeta API before uploading & inferring all images.

In [4]:
import os
import requests

In [5]:
NVCF_TOKEN="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"
UPLOAD_URL="https://api.nvcf.nvidia.com/v2/nvcf/assets"

In [6]:
dgx_dir = "/dgx1nas1/storage/data/jess/phenom_beta/tiff/2024_01_17_B7A1R1_P1T1__2024_01_17T08_35_58_Measurement_1"
tfs = os.listdir(dgx_dir)[0:2]
tfs

['r12c13f03p01-ch1sk1fk1fl1.tiff', 'r09c05f04p01-ch1sk1fk1fl1.tiff']

In [7]:
asset_ids = []
for tiff in tfs:
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {NVCF_TOKEN}",
        "accept": "application/json",
    }

    asset_description = {"contentType": "image/tiff", "description": "phenomics site channel"}
    s3_headers = {
        "x-amz-meta-nvcf-asset-description": "phenomics site channel",
        "content-type": "image/tiff",
    }

    response = requests.post(
        UPLOAD_URL, headers=headers, json=asset_description, timeout=30
    )
    response.raise_for_status()

    # Note: Asset upload and download URLs have a TTL of 1 hour.
    asset_url = response.json()["uploadUrl"]
    asset_id = response.json()["assetId"]

    # we will send binary data similar to CURL
    input_data_bytes = open(f"{dgx_dir}/{tiff}", "rb")
    response = requests.put(
        asset_url,
        data=input_data_bytes,
        headers=s3_headers,
        timeout=300,
    )
    response.raise_for_status()

    asset_ids.append(asset_id)

In [8]:
asset_ids

['9868b212-8d4d-4f60-aafd-0338e9467770',
 '432f8914-30a1-46ed-957f-f73b6665d0f9']

In [9]:
FUNCTION_ID = "7db32b36-ec04-43a6-a78f-1d8296accd8d"
FUNCTION_VERSION_ID = "3d73b252-008d-4469-b4c3-b25b9cbec654"
INVOKE_URL=f"https://api.nvcf.nvidia.com/v2/nvcf/exec/functions/{FUNCTION_ID}/versions/{FUNCTION_VERSION_ID}"

In [10]:
# prepare HTTP request
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {NVCF_TOKEN}",
}

payload = {
"requestHeader": {
    "inputAssetReferences": asset_ids,
},
"requestBody": {
    "inputs": [
    {
        "name": "scale_factor",
        "shape": [1],
        "datatype": "FP32",
        "data": [1],
    },
    ],
    "outputs": [{"name": "data"},
    ],
},
}

response = requests.post(
    INVOKE_URL,
    headers=headers,
    json=payload,
    timeout=30,
)

In [11]:
response.raise_for_status()

In [12]:
request_id = response.json()["reqId"]

POLL_URL=f"https://api.nvcf.nvidia.com/v2/nvcf/exec/status/{request_id}"

# Polling for result
while response.status_code == 202:
    response = requests.request("GET", POLL_URL, headers=headers, timeout=30)
    response.raise_for_status()

In [13]:
# figure out correct shape
embedding = response.json()["response"]["outputs"][0]["data"]
print(embedding)


[-0.4663783013820648, 1.5743474960327148, -1.3169991970062256, 0.7889142036437988, -1.0840667486190796, -1.229878306388855, 0.3356708288192749, 0.30798864364624023, -0.7087084054946899, 0.7916282415390015, -1.0257841348648071, -0.4484088718891144, -1.1647984981536865, -0.4417360723018646, 0.329010546207428, 0.189027801156044, -0.2481393665075302, 0.10677986592054367, 0.5094423294067383, 0.36603036522865295, 0.8648768663406372, 0.3285004198551178, 2.123030185699463, 0.29715389013290405, 0.16609564423561096, -0.20658357441425323, -0.6943368315696716, -0.00234298431314528, 0.34110379219055176, 2.016455888748169, -0.21789827942848206, -1.1875908374786377, 3.0305275917053223, -0.03788803890347481, -0.1775447130203247, -0.2389983981847763, 0.5339578986167908, -0.1738017201423645, 1.485863208770752, -0.32015666365623474, -0.390191912651062, 0.24160733819007874, 0.449571430683136, 0.21753787994384766, -3.3408799171447754, -0.2953193187713623, -0.25955474376678467, 0.20327281951904297, -0.59780

In [15]:

# Reshape into matrix like this:
import numpy as np
output = response.json()["response"]["outputs"][0]
crop_by_embedding_matrix = np.reshape(output["data"], output["shape"])

In [16]:
crop_by_embedding_matrix.shape

(16, 384)

In [17]:
    # get metadata
    plate_id = "P1"
    well_id = "W1"
    site_id = "S1"
    metadata = {
        "Plate": [plate_id] * 16,
        "Well": [well_id] * 16,
        "Site": [site_id] * 16,
        "Crop": [f"c{i+1}" for i in range(16)],
    }
    feature_columns = [f"f_{i+1:03d}" for i in range(384)]

In [19]:
import polars as pl
df = pl.DataFrame({
    **metadata,
    **{feature_columns[i]: crop_by_embedding_matrix[:, i] for i in range(384)}
})

## Delete assets from NVIDIA



In [ ]:
import requests

NVCF_TOKEN="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"
assets_URL = "https://api.nvcf.nvidia.com/v2/nvcf/assets"

In [ ]:
headers = {
    "Authorization": f"Bearer {NVCF_TOKEN}",
}
response = requests.request("GET", assets_URL, headers=headers, timeout=30)
assets = response.json()['assets']
for asset in tqdm(assets):
    #requests.request("DELETE", f"{assets_URL}/{asset['assetId']}", headers=headers)

## Check embeddings

In [1]:
import numpy as np
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq
import requests
from tqdm import tqdm

dgx_dir="/dgx1nas1/storage/data/jess/phenom_beta"

function_id = "7db32b36-ec04-43a6-a78f-1d8296accd8d"
version_id = "3d73b252-008d-4469-b4c3-b25b9cbec654"
nvcf_token = "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"
invoke_url = f"https://api.nvcf.nvidia.com/v2/nvcf/exec/functions/{function_id}/versions/{version_id}"

embedding_dir = "/dgx1nas1/storage/data/jess/phenom_beta/embeddings"

    # Define schema for embedding parquet
fields = [
    pa.field("Metadata_Plate", pa.string()),
    pa.field("Metadata_Well", pa.string()),
    pa.field("Metadata_Site", pa.string()),
    pa.field("Metadata_Crop", pa.string()),
] + [
    pa.field(f"f_{i+1:03d}", pa.float64()) for i in range(384)
]
dat_schema = pa.schema(fields)

In [4]:
plate = "2024_01_17_B7A1R1_P1T1__2024_01_17T08_35_58_Measurement_1"

id_df = pl.read_csv(f"{dgx_dir}/nvidia_assets/{plate}.csv").with_columns(
    pl.lit(plate).alias("plate"),
    pl.col("tiff_file").str.replace("f0.*", "").alias("well"),
    pl.col("tiff_file").str.replace("p01.*", "").alias("site_id"),
    pl.col("tiff_file").str.replace("p01.*", "").str.slice(6,3).alias("site"),
)
sites = id_df.select("site_id").to_series().unique().to_list()

In [6]:
sites = sites[0:10]

In [8]:
writer = pq.ParquetWriter(f"{embedding_dir}/{plate}_test.parquet", dat_schema, compression="gzip")

In [9]:
for site in tqdm(sites):

    # get asset ids for all channels in site
    site_df = id_df.filter(pl.col("site_id") == site)

    # get metadata
    plate_id = site_df.select("plate")[0].item()
    well_id = site_df.select("well")[0].item()
    site_id = site_df.select("site")[0].item()
    metadata = {
        "Metadata_Plate": [plate_id] * 16,
        "Metadata_Well": [well_id] * 16,
        "Metadata_Site": [site_id] * 16,
        "Metadata_Crop": [f"c{i+1}" for i in range(16)],
    }
    feature_columns = [f"f_{i+1:03d}" for i in range(384)]

    # prepare HTTP request
    asset_ids = site_df.select("asset_id").to_series().unique().to_list()
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {nvcf_token}",
    }

    payload = {
    "requestHeader": {
        "inputAssetReferences": asset_ids,
    },
    "requestBody": {
        "inputs": [
        {
            "name": "scale_factor",
            "shape": [1],
            "datatype": "FP32",
            "data": [1],
        },
        ],
        "outputs": [{"name": "data"},
        ],
    },
    }

    response = requests.post(
        invoke_url,
        headers=headers,
        json=payload,
        timeout=30,
    )
    response.raise_for_status()


    request_id = response.json()["reqId"]
    poll_url=f"https://api.nvcf.nvidia.com/v2/nvcf/exec/status/{request_id}"

    # Polling for result
    while response.status_code == 202:
        response = requests.request("GET", poll_url, headers=headers, timeout=30)
        response.raise_for_status()

    # reshape to crop x embedding
    output = response.json()["response"]["outputs"][0]
    embedding = np.reshape(output["data"], output["shape"])

    # append each response to larger array
    dat = pl.DataFrame({
        "Metadata_Plate": pl.Series(metadata["Metadata_Plate"], dtype=pl.Utf8),
        "Metadata_Well": pl.Series(metadata["Metadata_Well"], dtype=pl.Utf8),
        "Metadata_Site": pl.Series(metadata["Metadata_Site"], dtype=pl.Utf8),
        "Metadata_Crop": pl.Series(metadata["Metadata_Crop"], dtype=pl.Utf8),
        **{feature_columns[i]: embedding[:, i] for i in range(384)},
    }).to_pandas()

    writer.write_table(pa.Table.from_pandas(dat, schema=dat_schema))

100%|██████████| 10/10 [00:26<00:00,  2.63s/it]


In [10]:
writer.close()